In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install pandas

In [ ]:
!pip install joblib
!pip install scikit-learn

In [ ]:
# =========================
# STEP 0 — Setup (no histograms; removed Abs_Burst_Peak_List)
# =========================
# Optional installs (only if missing):
# pip install numpy pandas scikit-learn xgboost lightgbm joblib pyarrow fastparquet

import os, re, ast, json, math, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight

# ---- Paths / config ----
DATA_PATH = "/Users/rohan/Documents/Research/distrubution_data_ml/Compiled_Networks.csv"
LABEL_COL = "NeuronType"

# ONLY these four list columns now (Abs_Burst_Peak_List removed)
columns_to_process = [
    "Burst_Peak_List",
    "Burst_Times_List",      # used for timing, L, bursts/min
    "IBI_List",
    "SpikesPerBurst_List",
]

TIME_COL = "Burst_Times_List"

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)

# Each row = a recording (no unique id present) → synthesize one
df = df.reset_index(drop=True)
df["recording_id"] = [f"row_{i:06d}" for i in range(len(df))]

# Sanity checks
missing = [c for c in [LABEL_COL] + columns_to_process if c not in df.columns]
assert not missing, f"Missing columns in CSV: {missing}"

df[[LABEL_COL] + columns_to_process].head()


In [ ]:
# ---- Input paths ----
DATA_PATH = "/Users/rohan/Documents/Research/distrubution_data_ml/Compiled_Networks.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# =========================
# STEP 1 — Parse list columns into numeric arrays (robust to missing 'recording_id')
# =========================
import ast, re
import numpy as np
import pandas as pd

def parse_list_cell(x):
    if pd.isna(x):
        return np.array([], dtype=float)
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.array(x, dtype=float)
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.array([], dtype=float)
    # Try Python/JSON literal like "[1,2,3]"
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple, np.ndarray)):
            return np.array(v, dtype=float)
    except Exception:
        pass
    # Fallback: comma/space separated
    try:
        toks = [t for t in re.split(r"[,\s]+", s) if t != ""]
        return np.array([float(t) for t in toks], dtype=float)
    except Exception:
        return np.array([], dtype=float)

# --- normalize and find/standardize 'recording_id' ---
norm_map = {c.lower().strip(): c for c in df.columns}
candidate_keys = [
    "recording_id", "recordingid", "rec_id", "record_id", "recordid",
    "recording id", "record id", "recording"  # last two are looser matches
]
rec_col = None
for k in candidate_keys:
    if k in norm_map:
        rec_col = norm_map[k]
        break

df = df.copy()
if rec_col is None:
    # No recording id present — create one
    df["recording_id"] = np.arange(len(df))
else:
    if rec_col != "recording_id":
        df.rename(columns={rec_col: "recording_id"}, inplace=True)

# --- make sure columns_to_process exists and only includes present columns ---
# (If columns_to_process already defined earlier, this just filters it.)
present_cols = [c for c in columns_to_process if c in df.columns]
missing_cols = [c for c in columns_to_process if c not in df.columns]
if missing_cols:
    print(f"[WARN] Skipping missing columns: {missing_cols}")

# --- parse the list-like columns ---
parsed = {c: [] for c in present_cols}
rec_ids = df["recording_id"].tolist()

for _, row in df.iterrows():
    for c in present_cols:
        parsed[c].append(parse_list_cell(row[c]))


# Printing the above changes

In [ ]:
# --- parse the list-like columns ---
parsed = {c: [] for c in present_cols}
rec_ids = df["recording_id"].tolist()

for _, row in df.iterrows():
    for c in present_cols:
        parsed[c].append(parse_list_cell(row[c]))

# --- print parsed results for inspection ---
for i, rec_id in enumerate(rec_ids[:5]):  # limit to first 5 rows so output isn’t huge
    print(f"\nRecording ID: {rec_id}")
    for c in present_cols:
        print(f"  {c}: {parsed[c][i]}")


In [ ]:
# Remove histograms and re run the model to see if there is an improvement first burst time and last burst time

In [ ]:
# # =========================
# # STEP 2 — Global histogram bins per feature
# # =========================
# def global_edges(all_arrays, n_bins=N_BINS):
#     """
#     Build a single set of histogram bin edges for one feature,
#     shared by ALL recordings. This lets every recording's histogram
#     be comparable (same bin edges for everyone).

#     Parameters
#     ----------
#     all_arrays : list[np.ndarray]
#         For one feature (e.g., IBI_List), this is a list of 1D arrays,
#         one array per recording.
#     n_bins : int
#         Number of histogram bins to produce.

#     Returns
#     -------
#     np.ndarray
#         Array of length (n_bins + 1) with bin edges (monotonically increasing).
#     """
#     # Keep only proper, non-empty numpy arrays (skip Nones/empties)
#     vals = [a for a in all_arrays if isinstance(a, np.ndarray) and a.size]
#     if not vals:
#         # If literally no data at all for this feature, return a safe default
#         # edges from 0 to 1 split into n_bins bins.
#         return np.linspace(0, 1, n_bins + 1)

#     # Concatenate values from all recordings to find a global min/max range
#     concat = np.concatenate(vals)

#     # Compute global min/max while ignoring NaNs
#     vmin, vmax = float(np.nanmin(concat)), float(np.nanmax(concat))

#     # If min/max are not finite (all NaNs?) or equal (constant feature),
#     # build a small padded range so bins aren't degenerate
#     if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
#         # Fallback to a basic 0..1 range if min/max are not finite
#         vmin = vmin if np.isfinite(vmin) else 0.0
#         vmax = vmax if np.isfinite(vmax) else 1.0
#         # Add a tiny pad on both sides so we can still create n_bins intervals
#         return np.linspace(vmin - 0.5, vmax + 0.5, n_bins + 1)

#     # Normal case: equal-width bins spanning [vmin, vmax]
#     return np.linspace(vmin, vmax, n_bins + 1)

# # Build a dict: for each feature column, compute its shared bin edges once.
# # Example: hist_bins["IBI_List"] is a 1D array of edges used for that feature's histograms.
# hist_bins = {c: global_edges(parsed[c], N_BINS) for c in columns_to_process}


In [ ]:
# =========================
# STEP 3 — Feature helpers (robust stats only)
# =========================
def median_absolute_deviation(x):
    if x.size == 0:
        return np.nan
    med = np.nanmedian(x)
    return float(np.nanmedian(np.abs(x - med)))

def robust_stats(x):
    if x.size == 0:
        return {k: np.nan for k in [
            "mean","std","median","mad","min","q10","q25","q50","q75","q90","max","skew","kurtosis"
        ]}
    s = pd.Series(x, dtype=float)
    return {
        "mean": float(s.mean()),
        "std": float(s.std(ddof=1)),
        "median": float(s.median()),
        "mad": float(median_absolute_deviation(s.values)),
        "min": float(s.min()),
        "q10": float(s.quantile(0.10)),
        "q25": float(s.quantile(0.25)),
        "q50": float(s.quantile(0.50)),
        "q75": float(s.quantile(0.75)),
        "q90": float(s.quantile(0.90)),
        "max": float(s.max()),
        "skew": float(s.skew()),
        "kurtosis": float(s.kurtosis()),
    }


In [ ]:
# =========================
# STEP 4 — Build per-recording features (counts/timing + robust stats only)
# =========================
FEATURE_ROWS = []

for i, rec_id in enumerate(rec_ids):
    # Timing & L from burst times
    times = parsed.get(TIME_COL, [np.array([], dtype=float)])[i]
    times = np.sort(times) if times.size else times

    # L: prefer count of times; fallback to max length among other lists
    fallback_lengths = [parsed[c][i].size for c in columns_to_process if parsed[c][i] is not None]
    L = int(times.size) if times.size else (int(max(fallback_lengths)) if fallback_lengths else 0)
    first_t = float(times[0]) if times.size else np.nan
    last_t  = float(times[-1]) if times.size else np.nan
    span    = float(max(0.0, last_t - first_t)) if times.size else np.nan
    bursts_per_min = (L / (span/60.0)) if (times.size and span > 0) else np.nan

    # No per-burst durations in provided columns → leave NaN (will be imputed + flagged)
    feat = {
        "recording_id": rec_id,
        "L": L,
        "bursts_per_min": bursts_per_min,
        "first_burst_time": first_t,
        "last_burst_time": last_t,
        "approx_recording_span_sec": span,
        "has_bursts": 1 if L > 0 else 0,
    }

    # Robust stats for each list feature (NO histograms)
    for c in columns_to_process:
        x = parsed[c][i]
        for k, v in robust_stats(x).items():
            feat[f"{c}__{k}"] = v

    FEATURE_ROWS.append(feat)

feat_table = pd.DataFrame(FEATURE_ROWS)
feat_table.head()


In [ ]:
import os

assert "feat_table" in globals(), "Run Step 4 first to build `feat_table`."

OUT_DIR = "exports"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "features_step4_nohist.csv")

# Save CSV only
feat_table.to_csv(OUT_CSV, index=False)  # optionally add float_format="%.6g"
print(f"Saved CSV:\n- {OUT_CSV}")

# Quick summary
print("\nStep-4 feature table shape (rows, cols):", feat_table.shape)

cols = list(feat_table.columns)
print("\nFirst 40 columns:\n", cols[:40])
if len(cols) > 40:
    print("... (total columns:", len(cols), ")")

na_rate = feat_table.isna().mean().sort_values(ascending=False)
print("\nTop 15 columns by NA rate (pre-impute):")
print(na_rate.head(15))

display(feat_table.head(5))


In [ ]:
# =========================
# STEP 5 — L=0-safe: missingness indicators + median impute
# =========================
X = feat_table.copy()

# Add __isnan flags for numeric columns (except the binary 'has_bursts')
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c]) and c != "has_bursts"]
for c in num_cols:
    X[f"{c}__isnan"] = X[c].isna().astype(int)

# Median-impute numerics
for c in num_cols:
    med = float(np.nanmedian(X[c].values)) if X[c].notna().any() else 0.0
    X[c] = X[c].fillna(med)

# Keep ID for reference but don't feed it to the model
X = X[["recording_id"] + [c for c in X.columns if c != "recording_id"]]

# Attach label (NeuronType) aligned by row order
y = df[LABEL_COL].copy()

print("Shapes → X:", X.shape, " y:", y.shape)
X.head()


In this step, we first remove rows with missing labels so that every training example has a valid target. Next, we separate out the recording IDs (kept for reference only) and drop them from the feature matrix since they are not predictive. The neuron type labels are then encoded into integers using LabelEncoder, making them usable by machine learning models. We perform a stratified train/test split (75% training, 25% testing) to ensure that class proportions remain balanced across sets. To handle class imbalance, we compute class weights where rarer classes receive higher weight and build a sample_weight array aligned with the training data. Finally, we print the classes and their corresponding weights to verify everything is set up correctly before training.

In [ ]:
# =========================
# STEP 6 — Train/test split & label encoding
# =========================
# Drop rows with missing labels (if any)
mask = y.notna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

# Keep the id aside and remove from features
recording_ids = X["recording_id"].tolist()
X_model = X.drop(columns=["recording_id"])

# Encode NeuronType
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Stratified split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_model, y_enc, test_size=0.25, random_state=42, stratify=y_enc
)

# Class weights to help imbalance
classes = np.unique(y_tr)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
class_weight_map = {cls: w for cls, w in zip(classes, class_weights)}
sample_weight_tr = np.array([class_weight_map[c] for c in y_tr], dtype=float)

print("Classes:", list(le.classes_))
print("Class weights:", class_weight_map)


In [ ]:
!pip install lightgbm
!pip install xgboost

In [ ]:
# =========================
# STEP 7 — Train multiple models; pick best by macro-F1
# =========================
models = {}
reports = {}

# 7A) LightGBM (small-data friendly)
try:
    from lightgbm import LGBMClassifier
    lgbm = LGBMClassifier(
        objective="multiclass",
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=5,
        min_data_in_leaf=5,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=1,
        reg_lambda=1e-2,
        random_state=42,
        force_col_wise=True,
    )
    lgbm.fit(X_tr.astype(np.float32), y_tr, sample_weight=sample_weight_tr)
    y_pred = lgbm.predict(X_te.astype(np.float32))
    f1m = f1_score(y_te, y_pred, average="macro")
    models["LightGBM"] = lgbm
    reports["LightGBM"] = {
        "f1_macro": f1m,
        "accuracy": accuracy_score(y_te, y_pred),
        "y_pred": y_pred
    }
    print(f"LightGBM → F1_macro={f1m:.3f}  Acc={reports['LightGBM']['accuracy']:.3f}")
except Exception as e:
    print("LightGBM not available:", e)

# 7B) XGBoost (if available)
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        n_estimators=700,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        tree_method="hist"
    )
    xgb.fit(X_tr.astype(np.float32), y_tr, sample_weight=sample_weight_tr)
    y_pred = xgb.predict(X_te.astype(np.float32))
    f1m = f1_score(y_te, y_pred, average="macro")
    models["XGBoost"] = xgb
    reports["XGBoost"] = {
        "f1_macro": f1m,
        "accuracy": accuracy_score(y_te, y_pred),
        "y_pred": y_pred
    }
    print(f"XGBoost  → F1_macro={f1m:.3f}  Acc={reports['XGBoost']['accuracy']:.3f}")
except Exception as e:
    print("XGBoost not available:", e)

# 7C) MLP (always available; scale features)
mlp = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", MLPClassifier(hidden_layer_sizes=(256,128), alpha=1e-4, max_iter=700, random_state=42))
])
mlp.fit(X_tr, y_tr)
y_pred = mlp.predict(X_te)
f1m = f1_score(y_te, y_pred, average="macro")
models["MLP"] = mlp
reports["MLP"] = {
    "f1_macro": f1m,
    "accuracy": accuracy_score(y_te, y_pred),
    "y_pred": y_pred
}
print(f"MLP      → F1_macro={f1m:.3f}  Acc={reports['MLP']['accuracy']:.3f}")

# Pick best
best_name = max(reports, key=lambda k: reports[k]["f1_macro"])
best_model = models[best_name]
best_pred = reports[best_name]["y_pred"]
print("\nBEST MODEL:", best_name, " | F1_macro:", reports[best_name]["f1_macro"], " | Acc:", reports[best_name]["accuracy"])


## Printing the F1_macro and Accuracy for all 3 models XGBoost, MLP and LightGBM

In [ ]:
print("\n=== Model Performance Summary ===")
for name, rep in reports.items():
    print(f"{name:8s} → F1_macro={rep['f1_macro']:.3f} | Acc={rep['accuracy']:.3f}")

print("\n=== Best Model ===")
print(f"{best_name} → F1_macro={reports[best_name]['f1_macro']:.3f} | Acc={reports[best_name]['accuracy']:.3f}")


### Below code summarizes the best model's performance

In [ ]:
# =========================
# STEP 8 — Report for best model (confusion matrix + per-class metrics)
# =========================
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print(f"\n=== Evaluation for Best Model: {best_name} ===")

labels = list(le.classes_)
cm = confusion_matrix(y_te, best_pred, labels=np.arange(len(labels)))

print("Label order:", labels)
print("Confusion matrix (rows=true, cols=pred):\n", cm)

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
np.set_printoptions(precision=3, suppress=True)
print("Row-normalized (%):\n", np.round(100 * cm_norm, 1))

print("\nClassification report:\n")
print(classification_report(y_te, best_pred, target_names=labels, digits=3))


# what the above results mean?
Your best model (XGBoost) reached about 70% accuracy on the test set, which is well above the 47% majority-class baseline. Looking at the confusion matrix and report:

FxHET: The model correctly identified half of FxHET samples (recall = 50%). Precision was ~64%, meaning when it predicted FxHET, it was right about two-thirds of the time.

MxHEMI: Performance was strongest here — 71% recall and 71% precision, showing the model handled this smaller class fairly well.

MxWT: The largest class had the highest recall (86%), meaning most MxWT samples were correctly recognized. Precision was ~73%, so there were still some false positives.

Overall, the macro-averaged F1 score (~0.69) shows the model is reasonably balanced across all three neuron types, not just favoring the majority. In short: the model is clearly learning useful patterns, but FxHET is harder to classify reliably compared to the others.

# OLD RESULT WITH abs burst

FxHET (support=18): model got 7 right, misclassified 3 as MxHEMI and 8 as MxWT.
Row-normalized: 38.9% correct, 16.7% → MxHEMI, 44.4% → MxWT.
→ FxHET is the hardest; the model often confuses it with MxWT.

MxHEMI (support=7): model got 6 right, only 1 mistake (as MxWT).
Row-normalized: 85.7% correct → best recall.

MxWT (support=22): model got 16 right, 6 were predicted as FxHET.
Row-normalized: 72.7% correct.


Your report:

FxHET: precision 0.538, recall 0.389, F1 0.452 → biggest pain point (lots of misses to MxWT).

MxHEMI: precision 0.667, recall 0.857, F1 0.750 → strong.

MxWT: precision 0.640, recall 0.727, F1 0.681 → decent.

In [ ]:
!pip install matplotlib

# heatmap visualization of the confusion matrix for best model in this case XGBoost

In [ ]:
import matplotlib.pyplot as plt

# Plot confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues")

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Percentage", rotation=-90, va="bottom")

# Tick labels
ax.set(
    xticks=np.arange(len(labels)),
    yticks=np.arange(len(labels)),
    xticklabels=labels,
    yticklabels=labels,
    ylabel="True label",
    xlabel="Predicted label",
    title=f"Confusion Matrix Heatmap ({best_name})"
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

# Write the % values inside the cells
thresh = cm_norm.max() / 2.
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(
            j, i, f"{cm_norm[i, j]*100:.1f}",
            ha="center", va="center",
            color="white" if cm_norm[i, j] > thresh else "black"
        )

plt.tight_layout()
plt.show()


# Cross Validation of the best model

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
import numpy as np

# Convert to numpy for CV
X_np = X_model.astype(np.float32).values
y_np = y_enc.copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, f1ms = [], []

for tr, te in skf.split(X_np, y_np):
    xtr, xte = X_np[tr], X_np[te]
    ytr, yte = y_np[tr], y_np[te]

    # Sample weights (to handle class imbalance)
    sw = np.array([class_weight_map[c] for c in ytr], dtype=float)

    # Define XGBoost model
    xgb = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        n_estimators=700,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=2.0,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=0.5,
        random_state=42,
        tree_method="hist"
    )

    # Train on fold
    xgb.fit(xtr, ytr, sample_weight=sw)

    # Predict on validation fold
    yhat = xgb.predict(xte)

    # Collect metrics
    accs.append(accuracy_score(yte, yhat))
    f1ms.append(f1_score(yte, yhat, average="macro"))

print("XGBoost 5-fold CV — Acc: {:.3f}±{:.3f} | Macro-F1: {:.3f}±{:.3f}"
      .format(np.mean(accs), np.std(accs), np.mean(f1ms), np.std(f1ms)))


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import numpy as np

# Use the SAME data arrays for both models
X_np = X_model.values   # leave NaNs; MLP pipeline will impute
y_np = y_enc.copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs_xgb, f1s_xgb = [], []
accs_mlp, f1s_mlp = [], []

for tr, te in skf.split(X_np, y_np):
    xtr, xte = X_np[tr], X_np[te]
    ytr, yte = y_np[tr], y_np[te]

    # sample weights (same as you used before)
    sw = np.array([class_weight_map[c] for c in ytr], dtype=float)

    # ---- XGBoost (handles NaNs natively)
    xgb = XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        n_estimators=700, learning_rate=0.05,
        max_depth=6, min_child_weight=2.0,
        subsample=0.9, colsample_bytree=0.9,
        reg_lambda=0.5, random_state=42, tree_method="hist"
    )
    xgb.fit(xtr, ytr, sample_weight=sw)
    yhat_x = xgb.predict(xte)
    accs_xgb.append(accuracy_score(yte, yhat_x))
    f1s_xgb.append(f1_score(yte, yhat_x, average="macro"))

    # ---- MLP (needs imputation inside pipeline)
    mlp = Pipeline([
        ("impute", SimpleImputer(strategy="median")),    # train-only median
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", MLPClassifier(hidden_layer_sizes=(256,128), alpha=1e-4,
                              max_iter=700, random_state=42))
    ])
    mlp.fit(xtr, ytr)
    yhat_m = mlp.predict(xte)
    accs_mlp.append(accuracy_score(yte, yhat_m))
    f1s_mlp.append(f1_score(yte, yhat_m, average="macro"))

print("XGB  5-fold — Acc: {:.3f}±{:.3f} | Macro-F1: {:.3f}±{:.3f}"
      .format(np.mean(accs_xgb), np.std(accs_xgb), np.mean(f1s_xgb), np.std(f1s_xgb)))
print("MLP  5-fold — Acc: {:.3f}±{:.3f} | Macro-F1: {:.3f}±{:.3f}"
      .format(np.mean(accs_mlp), np.std(accs_mlp), np.mean(f1s_mlp), np.std(f1s_mlp)))

# Optional: simple paired comparison on Macro-F1 across the 5 folds
diff = np.array(f1s_xgb) - np.array(f1s_mlp)
print("Per-fold Macro-F1 diffs (XGB - MLP):", np.round(diff, 4),
      " | mean diff =", np.round(diff.mean(), 4))


# Printing the second best model's results

In [ ]:
# === Metrics for the Second-Best Model ===
import numpy as np
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix, classification_report, log_loss
)

# Sort models by F1_macro (descending)
sorted_models = sorted(reports.items(), key=lambda kv: kv[1]["f1_macro"], reverse=True)

# Pick the second-best
second_name, second_report = sorted_models[1]
second_model = models[second_name]
second_pred = second_report["y_pred"]

print(f"\n=== Evaluation for Second-Best Model: {second_name} ===")

# Core metrics
acc = accuracy_score(y_te, second_pred)
f1m = f1_score(y_te, second_pred, average="macro")
print(f"{second_name} — Acc: {acc:.3f} | F1_macro: {f1m:.3f}")

# Optional: probability-based metric (log loss)
if hasattr(second_model, "predict_proba"):
    y_proba = second_model.predict_proba(X_te.astype(np.float32))
    ll = log_loss(y_te, y_proba, labels=np.arange(len(le.classes_)))
    print(f"LogLoss: {ll:.4f}")

# Confusion matrix
labels = list(le.classes_)
cm = confusion_matrix(y_te, second_pred, labels=np.arange(len(labels)))
print("Label order:", labels)
print("Confusion matrix (rows=true, cols=pred):\n", cm)

# Row-normalized (per-true-class recall breakdown)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
np.set_printoptions(precision=3, suppress=True)
print("Row-normalized (%):\n", np.round(100 * cm_norm, 1))

# Per-class precision/recall/F1
print(f"\nClassification report ({second_name}):\n")
print(classification_report(y_te, second_pred, target_names=labels, digits=3))


In [ ]:
# Class distribution in the full labeled set
print(pd.Series(le.inverse_transform(y_enc)).value_counts())

# Majority-class baseline accuracy on the test set
maj = np.bincount(y_tr).argmax()
maj_acc = (y_te == maj).mean()
print(f"Majority-class baseline (test): {maj_acc:.3f}")


# what does the above cell mean 

Your dataset contains three neuron types: MxWT (86 samples), FxHET (70 samples), and MxHEMI (30 samples). This shows the data is imbalanced, with MxWT being the largest group and MxHEMI the smallest. If you built a trivial model that always predicts the majority class (MxWT), it would achieve about 46.8% accuracy on the test set. This baseline highlights the minimum standard your models must beat—any model performing only slightly better than ~47% isn’t very useful, but models that score well above this baseline are genuinely learning meaningful patterns from the features.

Total = 86 + 70 + 30 = 186 samples overall → mild imbalance:

MxWT ≈ 46%

FxHET ≈ 38%

MxHEMI ≈ 16%

# SHAP Analysis

In [ ]:
!pip install shap

# Summary Plot
The summary plot shows the feature importance of each feature in the model

In [ ]:
print(list(X_model.columns))


In [ ]:
# --- Class-wise mean(|SHAP|) horizontal stacked bars (like your screenshot) ---

# If SHAP not installed yet:
# !pip install shap

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

# ---- 1) Build explainer + pick eval subset (fast & reproducible) ----
feature_names = list(X_model.columns)

bg_n = min(1000, len(X_tr))
te_n = min(1000, len(X_te))
X_bg   = X_tr.sample(n=bg_n, random_state=42).astype(np.float32)
X_eval = X_te.sample(n=te_n, random_state=42).astype(np.float32)

# Modern wrapper first; fall back to TreeExplainer for older SHAP
try:
    explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
    sv = explainer(X_eval)  # shap.Explanation
except Exception:
    explainer = shap.TreeExplainer(xgb)
    sv = explainer.shap_values(X_eval)  # list-of-arrays per class (old API)

# ---- 2) Compute mean(|SHAP|) per feature, per class (works for both APIs) ----
def classwise_mean_abs_shap(shap_values, feature_names, class_names=None):
    """
    Returns a DataFrame with columns: feature, Class 0, Class 1, ..., Total
    Compatible with:
      - list of arrays [n_classes x (n_samples, n_features)]
      - Explanation/ndarray with shape (n_samples, n_features, n_classes)
      - (rare) 2D array (n_samples, n_features) → treated as 1-class
    """
    if isinstance(shap_values, list):
        # Old API: list-of-arrays (one (n_samples, n_features) per class)
        per_class = [np.mean(np.abs(a), axis=0) for a in shap_values]
        mat = np.vstack(per_class)  # (n_classes, n_features)
    else:
        vals = shap_values.values if hasattr(shap_values, "values") else shap_values
        if vals.ndim == 3:
            # (n_samples, n_features, n_classes)
            mat = np.mean(np.abs(vals), axis=0).T  # -> (n_classes, n_features)
        elif vals.ndim == 2:
            # Single "class" case: (n_samples, n_features)
            mat = np.mean(np.abs(vals), axis=0, keepdims=True)  # (1, n_features)
        else:
            raise ValueError(f"Unexpected SHAP array shape: {vals.shape}")

    n_classes, n_features = mat.shape
    if class_names is None:
        class_names = [f"Class {i}" for i in range(n_classes)]
    col_names = list(class_names)

    df = pd.DataFrame(mat.T, columns=col_names)
    df.insert(0, "feature", feature_names)
    df["Total"] = df[col_names].sum(axis=1)
    return df.sort_values("Total", ascending=False).reset_index(drop=True)

class_names = list(getattr(le, "classes_", [f"Class {i}" for i in range(xgb.n_classes_) ]))
df_imp = classwise_mean_abs_shap(sv, feature_names, class_names)

# ---- 3) Plot top-K features as stacked horizontal bars ----
TOP_K = 35  # adjust as you like
plot_df = df_imp.head(TOP_K).iloc[::-1]  # reverse for top at top

fig, ax = plt.subplots(figsize=(10,12))
left = np.zeros(len(plot_df))
class_cols = [c for c in plot_df.columns if c not in ("feature", "Total")]

for c in class_cols:
    ax.barh(plot_df["feature"], plot_df[c].values, left=left, label=c)
    left += plot_df[c].values

ax.set_xlabel("mean(|SHAP value|) (average impact on model output magnitude)")
ax.legend(title="Class", loc="lower right")
ax.set_title("Class-wise mean(|SHAP|) per Feature")
plt.tight_layout()
plt.show()


# Summary plot of the label '0'
- Y-axis indicates the feature names in order of importance from top to bottom.
- X-axis represents the SHAP value, which indicates the degree of change in log odds.
- The color of each point on the graph represents the value of the corresponding feature, with red indicating high values and blue indicating low values.
- Each point represents a row of data from the original dataset.

In [ ]:
# Beeswarm for ALL classes (one plot per class), and also save PNGs

import numpy as np
import matplotlib.pyplot as plt
import shap

feature_names = list(X_model.columns)

# Use your existing eval/background or create them if missing
if "X_eval" not in globals():
    X_bg   = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    X_eval = X_te.astype(np.float32)

# Build explainer / SHAP values if not already available as `sv`
if "sv" not in globals():
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
        sv = explainer(X_eval)   # shap.Explanation
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)  # list per class (old API)

# Get class names (from your LabelEncoder)
class_names = list(le.classes_)

def get_sv_for_class(sv_obj, class_idx):
    """Return a 2D array (n_samples, n_features) for the given class index,
       handling both new and old SHAP APIs."""
    if isinstance(sv_obj, list):        # old API: list of arrays per class
        return sv_obj[class_idx]
    vals = sv_obj.values if hasattr(sv_obj, "values") else sv_obj
    if vals.ndim == 3:                  # (n_samples, n_features, n_classes)
        return vals[:, :, class_idx]
    return vals                         # already 2D (binary-collapsed)

# Loop over classes and plot
for ci, cname in enumerate(class_names):
    sv_class = get_sv_for_class(sv, ci)
    plt.figure(figsize=(9, max(6, len(feature_names)*0.35)))
    shap.summary_plot(
        sv_class,
        X_eval,
        feature_names=feature_names,
        show=False,
        max_display=len(feature_names)  # show ALL features
    )
    plt.title(f"SHAP Summary (Beeswarm) — class: {cname}")
    plt.tight_layout()
    # Save and show
    out_path = f"shap_beeswarm_{cname}.png"
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


In [ ]:
# ---- Compact per-class direction summary (sign + strength) ----
import numpy as np, pandas as pd, shap
from scipy.stats import spearmanr

feature_names = list(X_model.columns)

# Ensure eval/background + SHAP values exist
if "X_eval" not in globals():
    X_bg   = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    X_eval = X_te.astype(np.float32)

if "explainer" not in globals() or "sv" not in globals():
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
        sv = explainer(X_eval)                  # new API: Explanation
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)      # old API: list per class

def _sv_matrix_for_class(sv_obj, ci):
    if isinstance(sv_obj, list):
        return sv_obj[ci]                       # (n_samples, n_features)
    vals = sv_obj.values if hasattr(sv_obj, "values") else sv_obj
    return vals[:, :, ci] if vals.ndim == 3 else vals

def _mean_abs_shap_per_feat(mat):               # mat: (n_samples, n_features)
    return np.mean(np.abs(mat), axis=0)

classes = list(le.classes_)
for ci, cname in enumerate(classes):
    shap_mat = _sv_matrix_for_class(sv, ci)             # (n_samples, n_features)
    mas = _mean_abs_shap_per_feat(shap_mat)

    rows = []
    for j, feat in enumerate(feature_names):
        rho, _ = spearmanr(np.asarray(X_eval[feat]), shap_mat[:, j])
        rows.append((feat, mas[j], rho))
    df = pd.DataFrame(rows, columns=["feature", "mean_abs_shap", "spearman_value_vs_shap"])
    df.sort_values("mean_abs_shap", ascending=False, inplace=True)

    # Keep a compact view
    top = df.head(25)
    top["direction"] = np.where(top["spearman_value_vs_shap"] > 0, "higher → toward class", "higher → away from class")
    display(top[["feature", "mean_abs_shap", "spearman_value_vs_shap", "direction"]])

    out = f"shap_direction_summary_{cname}.csv"
    top.to_csv(out, index=False)
    print("Saved:", out)


In [ ]:
# --- Beeswarm for TOP-K features per class (compact) ---
import numpy as np, matplotlib.pyplot as plt, shap

TOP_K = 20  # change as you like

feature_names = list(X_model.columns)

# Ensure eval/background + SHAP values exist
if "X_eval" not in globals():
    X_bg   = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    X_eval = X_te.astype(np.float32)

if "explainer" not in globals() or "sv" not in globals():
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
        sv = explainer(X_eval)                  # new API
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)      # old API: list per class

classes = list(le.classes_)

def _sv_class(sv_obj, ci):
    # Return (n_samples, n_features) for class ci
    if isinstance(sv_obj, list):        # old API
        return sv_obj[ci]
    vals = sv_obj.values if hasattr(sv_obj, "values") else sv_obj
    return vals[:, :, ci] if vals.ndim == 3 else vals

for ci, cname in enumerate(classes):
    sv_c = _sv_class(sv, ci)                         # (n_samples, n_features)
    mas  = np.mean(np.abs(sv_c), axis=0)             # mean |SHAP| per feature
    top_idx = np.argsort(-mas)[:TOP_K]               # indices of top-K
    fnames = [feature_names[i] for i in top_idx]

    plt.figure(figsize=(9, max(6, TOP_K * 0.35)))
    shap.summary_plot(
        sv_c[:, top_idx],
        X_eval.iloc[:, top_idx],
        feature_names=fnames,
        show=False,
        max_display=TOP_K
    )
    plt.title(f"SHAP Beeswarm — Top {TOP_K} features — {cname}")
    plt.tight_layout()
    out = f"shap_beeswarm_top{TOP_K}_{cname}.png"
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", out)


# Waterfall Plot
Shows how each feature contributes to the difference between the model's base value and the output prediction for a specific instance.

In [ ]:
# --- SHAP Waterfall for ALL classes (compact) ---
import numpy as np, matplotlib.pyplot as plt, shap

FEATURES = list(X_model.columns)
IDX = 0                 # use the same test row for all classes
MOST_CONFIDENT = False  # set True to pick most-confident row per class
MAX_DISPLAY = 20

# Ensure eval/background + SHAP values exist
if "X_eval" not in globals():
    X_bg   = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    X_eval = X_te.astype(np.float32)

if "explainer" not in globals() or "sv" not in globals():
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=FEATURES)
        sv = explainer(X_eval)                 # new API: Explanation
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)     # old API: list per class

classes = list(le.classes_)
proba = xgb.predict_proba(X_eval)  # (n_samples, n_classes)

def explanation_for(idx, ci):
    x_row = X_eval.iloc[idx].values
    if isinstance(sv, list):  # old API
        vals = sv[ci][idx, :]
        base = explainer.expected_value
        base = float(base[ci] if hasattr(base, "__len__") else base)
        return shap.Explanation(values=vals, base_values=base, data=x_row, feature_names=FEATURES)
    vals  = sv.values
    basev = getattr(sv, "base_values", None)
    if vals.ndim == 3:  # (n, f, C)
        return shap.Explanation(values=vals[idx, :, ci],
                                base_values=float(basev[idx, ci]) if basev is not None else 0.0,
                                data=x_row, feature_names=FEATURES)
    return shap.Explanation(values=vals[idx, :],
                            base_values=float(basev[idx]) if basev is not None else 0.0,
                            data=x_row, feature_names=FEATURES)

for ci, cname in enumerate(classes):
    use_idx = int(np.argmax(proba[:, ci])) if MOST_CONFIDENT else IDX
    ex = explanation_for(use_idx, ci)
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(ex, max_display=MAX_DISPLAY, show=False)
    plt.title(f"Waterfall — sample {use_idx} — class: {cname}")
    plt.tight_layout()
    out = f"waterfall_sample{use_idx}_{cname}.png"
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", out)


In [ ]:
# For sample i and class ci, check that base + sum(SHAP) ~ model logit
i, ci = 0, 0  # sample 0, class index 0 (e.g., FxHET)
proba = xgb.predict_proba(X_eval)
# Recover logits from probs via log-softmax inverse (relative to an arbitrary offset)
# Easier: directly compute base+sum(shap) from your Explanation:
ex = shap.Explanation(values=sv.values[i,:,ci], base_values=sv.base_values[i,ci], 
                      data=X_eval.iloc[i].values, feature_names=list(X_model.columns))
approx_logit = ex.base_values + ex.values.sum()
approx_logit


## To see which features you can add to make one neuron go from HET classification to WT classification

In [ ]:
# assumes `sv` is the new Explanation with sv.values shaped (n, f, C) and
# `classes` has the class order; fallback: adapt if you're on the old API.
import numpy as np, pandas as pd

ci_wt  = classes.index("MxWT")     # adjust to your exact WT label
ci_het = classes.index("FxHET")    # adjust to your exact HET label

vals = sv.values                  # (n, f, C)
sh_wt  = vals[:, :, ci_wt]
sh_het = vals[:, :, ci_het]

df = pd.DataFrame({
    "feature": FEATURES,
    "mean_SHAP_WT": sh_wt.mean(0),
    "mean_abs_SHAP_WT": np.abs(sh_wt).mean(0),
    "mean_SHAP_HET": sh_het.mean(0),
    "mean_abs_SHAP_HET": np.abs(sh_het).mean(0),
    "mean_SHAP_HET_minus_WT": sh_het.mean(0) - sh_wt.mean(0)
}).sort_values("mean_SHAP_HET_minus_WT", ascending=False)

# Top drivers toward HET vs WT:
top_to_HET = df.head(20)
top_to_WT  = df.tail(20).iloc[::-1]
print("Toward HET:\n", top_to_HET[["feature","mean_SHAP_HET_minus_WT"]])
print("Toward WT:\n",  top_to_WT[["feature","mean_SHAP_HET_minus_WT"]])


# Force Plot
We will examine the first sample in the testing set to determine which features contributed to the "0" result. To do this, we will utilize a force plot and provide the expected value, SHAP value, and testing sample.

### Most confident sample

In [ ]:
proba = xgb.predict_proba(X_eval)  # (n_samples, n_classes)
class_names = list(le.classes_)

most_confident_idx = {class_names[ci]: int(np.argmax(proba[:, ci]))
                      for ci in range(proba.shape[1])}
most_confident_idx


In [ ]:
# ---- Compact SHAP force plot for multiclass XGBoost ----
import numpy as np, matplotlib.pyplot as plt, shap

# Ensure SHAP explainer/values exist
feature_names = list(X_model.columns)
if "X_eval" not in globals():
    X_bg   = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    X_eval = X_te.astype(np.float32)

if "explainer" not in globals() or "sv" not in globals():
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
        sv = explainer(X_eval)    # new API
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)  # old API: list per class

class_names = list(le.classes_)

def _row_sv_and_base(sv_obj, expl, row_i, class_i):
    if isinstance(sv_obj, list):                       # old API
        row_vals = sv_obj[class_i][row_i, :]
        base = expl.expected_value
        base = float(base[class_i] if hasattr(base, "__len__") else base)
        return row_vals, base
    vals  = sv_obj.values
    basev = getattr(sv_obj, "base_values", None)
    if vals.ndim == 3:                                 # (n, f, C)
        return vals[row_i, :, class_i], float(basev[row_i, class_i])
    return vals[row_i, :], float(basev[row_i])

def force_plot(idx=0, class_name=None, save_prefix="force"):
    ci = class_names.index(class_name) if class_name else 0
    row_vals, basev = _row_sv_and_base(sv, explainer, idx, ci)
    x_row_series = X_eval.iloc[idx]

    # Interactive HTML (recommended)
    shap.initjs()
    force_html = shap.plots.force(basev, row_vals, x_row_series.values, feature_names=feature_names)
    html_path = f"{save_prefix}_sample{idx}_{class_names[ci]}.html"
    shap.save_html(html_path, force_html)

    # Static PNG (use matplotlib=True and save current fig immediately)
    plt.figure(figsize=(12, 2.6))
    shap.plots.force(basev, row_vals, x_row_series.values, feature_names=feature_names, matplotlib=True)
    plt.title(f"Force — sample {idx} — class: {class_names[ci]}")
    plt.tight_layout()
    fig = plt.gcf()
    png_path = f"{save_prefix}_sample{idx}_{class_names[ci]}.png"
    fig.savefig(png_path, dpi=180, bbox_inches="tight")
    plt.show()

    print(f"Saved HTML: {html_path}\nSaved PNG : {png_path}")

# Example: one sample for each class (most confident per class)
proba = xgb.predict_proba(X_eval)
for ci, cname in enumerate(class_names):
    idx = int(np.argmax(proba[:, ci]))   # pick the test row with highest P(class)
    force_plot(idx=idx, class_name=cname, save_prefix="force")


# Decision Plot

In [ ]:
# --- SHAP decision plots for all NeuronTypes (compact) ---
import numpy as np, matplotlib.pyplot as plt, shap

feature_names = list(X_model.columns)

# Ensure eval/background + SHAP values exist
if "X_eval" not in globals():
    X_bg   = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    X_eval = X_te.astype(np.float32)

if "explainer" not in globals() or "sv" not in globals():
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
        sv = explainer(X_eval)                  # shap.Explanation (new API)
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)      # list per class (old API)

classes = list(le.classes_)

def _vals_base_for_class(sv_obj, ci):
    """Return (n_samples,n_features) shap matrix and a scalar base value for class ci."""
    if isinstance(sv_obj, list):  # old API
        vals = sv_obj[ci]
        base = getattr(explainer, "expected_value", 0.0)
        base = float(base[ci] if hasattr(base, "__len__") else base)
        return vals, base
    vals  = sv_obj.values
    basev = getattr(sv_obj, "base_values", None)
    if vals.ndim == 3:   # (n_samples, n_features, n_classes)
        return vals[:, :, ci], float(np.mean(basev[:, ci])) if basev is not None else 0.0
    return vals, float(np.mean(basev)) if basev is not None else 0.0

# Optional: subsample lines to keep the plot readable
N = min(200, len(X_eval))
rng = np.random.RandomState(42)
idx = rng.choice(len(X_eval), N, replace=False)

for ci, cname in enumerate(classes):
    shap_mat, base = _vals_base_for_class(sv, ci)
    shap_mat = shap_mat[idx, :]  # align to subsample
    plt.figure(figsize=(10,5))
    shap.decision_plot(base, shap_mat, feature_names=feature_names, link="logit", show=False)
    plt.title(f"Decision plot — {cname}")
    plt.tight_layout()
    out = f"decision_{cname}.png"
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", out)


# Visualizing the results in the space

In [ ]:
# =========================
# STEP 8 — Visualize model prediction space with PCA
# =========================
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1) Get class probabilities on the test set
# (works for LightGBM, XGBoost, and the MLP pipeline)
probs_te = best_model.predict_proba(X_te.astype(np.float32))  # shape (n_samples, n_classes)

# 2) Project 3D prob vectors -> 2D with PCA
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(probs_te)   # shape (n_samples, 2)

# 3) Plot: color by TRUE label, highlight misclassifications
label_names = le.classes_                        # ['FxHET','MxHEMI','MxWT'] in whatever order
n_classes = len(label_names)
colors = ["tab:red", "tab:blue", "tab:green"][:n_classes]

plt.figure(figsize=(6, 5))

for k in range(n_classes):
    mask = (y_te == k)
    # correct vs incorrect
    correct = mask & (best_pred == y_te)
    wrong   = mask & (best_pred != y_te)

    # correct points (filled)
    plt.scatter(
        Z[correct, 0], Z[correct, 1],
        s=35, alpha=0.8,
        c=colors[k],
        label=f"{label_names[k]} (correct)"
    )

    # misclassified points (same color, black edge)
    if np.any(wrong):
        plt.scatter(
            Z[wrong, 0], Z[wrong, 1],
            s=60, alpha=0.9,
            facecolors="none",
            edgecolors="k",
            linewidths=1.2,
            marker="o"
        )

plt.xlabel("PC1 of prediction space")
plt.ylabel("PC2 of prediction space")
plt.title(f"{best_name}: test recordings in model output space\n(PCA on predict_proba)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
print("\nBEST MODEL:", best_name, " | F1_macro:", reports[best_name]["f1_macro"], " | Acc:", reports[best_name]["accuracy"])


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=42)
Z_pca = pca.fit_transform(probs_te)   # (n_samples, 2)

plt.figure(figsize=(6, 5))
for k, cname in enumerate(label_names):
    mask = (y_true == k)
    plt.scatter(
        Z_pca[mask, 0], Z_pca[mask, 1],
        s=35, alpha=0.8, c=colors[k], label=cname
    )

plt.xlabel("PC1 of prediction space")
plt.ylabel("PC2 of prediction space")
plt.title("XGBoost: PCA on predict_proba")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# unpack probabilities
p_fx, p_mxh, p_mxwt = probs_te.T   # each shape (n_samples,)

# triangle vertices (equilateral)
A = np.array([0.0, 0.0])                     # FxHET corner
B = np.array([1.0, 0.0])                     # MxHEMI corner
C = np.array([0.5, np.sqrt(3)/2])            # MxWT corner

# barycentric -> Cartesian
XY = p_fx[:, None] * A + p_mxh[:, None] * B + p_mxwt[:, None] * C
x, y = XY[:, 0], XY[:, 1]

plt.figure(figsize=(6, 6))
for k, cname in enumerate(label_names):
    mask = (y_true == k)
    plt.scatter(x[mask], y[mask], s=35, alpha=0.8, c=colors[k], label=cname)

# draw triangle
plt.plot([A[0], B[0], C[0], A[0]], [A[1], B[1], C[1], A[1]], "k-")

plt.text(A[0] - 0.05, A[1] - 0.03, "FxHET", ha="right", va="top")
plt.text(B[0] + 0.05, B[1] - 0.03, "MxHEMI", ha="left",  va="top")
plt.text(C[0],        C[1] + 0.05, "MxWT",  ha="center", va="bottom")

plt.axis("equal")
plt.axis("off")
plt.title("XGBoost: prediction simplex (no PCA)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(
    n_components=2,
    perplexity=15,      # tweakable
    learning_rate="auto",
    init="pca",
    random_state=42,
)
Z_tsne = tsne.fit_transform(probs_te)   # (n_samples, 2)

plt.figure(figsize=(6, 5))
for k, cname in enumerate(label_names):
    mask = (y_true == k)
    plt.scatter(
        Z_tsne[mask, 0], Z_tsne[mask, 1],
        s=35, alpha=0.8, c=colors[k], label=cname
    )

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("XGBoost: t-SNE on predict_proba")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
pip install umap-learn

In [ ]:
import umap
import matplotlib.pyplot as plt

reducer = umap.UMAP(
    n_components=2,
    random_state=42,
    n_neighbors=15,     # similar role to t-SNE perplexity
    min_dist=0.1,
)
Z_umap = reducer.fit_transform(probs_te)   # (n_samples, 2)

plt.figure(figsize=(6, 5))
for k, cname in enumerate(label_names):
    mask = (y_true == k)
    plt.scatter(
        Z_umap[mask, 0], Z_umap[mask, 1],
        s=35, alpha=0.8, c=colors[k], label=cname
    )

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("XGBoost: UMAP on predict_proba")
plt.legend()
plt.tight_layout()
plt.show()


## What this is doing (step by step)
probs_te = best_model.predict_proba(X_te.astype(np.float32))
### For each test recording, XGBoost returns
[𝑃(FxHET∣𝑥),𝑃(MxHEMI∣𝑥),𝑃(MxWT∣𝑥)]
[P(FxHET∣x), P(MxHEMI∣x), P(MxWT∣x)]
So probs_te has shape (n_test_samples, 3).
### PCA on those 3D vectors
Compresses the 3-dimensional probability vectors into 2D while preserving as much variance as possible.
Z[:, 0] and Z[:, 1] are coordinates in “model-output space.”
### Scatter plot
- Each point = one recording in the test set.
- Color = true NeuronType.
- If a class clusters nicely, the model gives similar probability patterns for that class.
- If FxHET spreads between MxHEMI and MxWT regions, that matches the confusion we saw earlier.

In [ ]:
# =========================
# STEP 9 — Build "SHAP space" via PCA on SHAP values
# =========================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

feature_names = list(X_model.columns)
class_names = list(le.classes_)   # ['FxHET', 'MxHEMI', 'MxWT']

# 1) Define evaluation set and ensure we have SHAP values for it
X_eval = X_te.astype(np.float32)
y_true = y_te

# If you already built `explainer` and `sv`, you can reuse them.
# Otherwise, rebuild here for clarity:
if "sv" not in globals():
    X_bg = X_tr.sample(min(1000, len(X_tr)), random_state=42).astype(np.float32)
    try:
        explainer = shap.Explainer(xgb, X_bg, feature_names=feature_names)
        sv = explainer(X_eval)   # new API: shap.Explanation
    except Exception:
        explainer = shap.TreeExplainer(xgb)
        sv = explainer.shap_values(X_eval)  # old API: list-of-arrays

def shap_array_for_class(sv_obj, class_idx):
    """
    Return a 2D array of SHAP values (n_samples, n_features) for a given class index,
    handling both the new shap.Explanation API and the old list-of-arrays API.
    """
    if isinstance(sv_obj, list):  # old API
        return sv_obj[class_idx]  # (n_samples, n_features)

    vals = sv_obj.values if hasattr(sv_obj, "values") else sv_obj
    if vals.ndim == 3:
        # (n_samples, n_features, n_classes)
        return vals[:, :, class_idx]
    elif vals.ndim == 2:
        # Already 2D (binary or single-output case)
        return vals
    else:
        raise ValueError(f"Unexpected SHAP array shape: {vals.shape}")

# 2) Build a SHAP matrix for the class the model actually predicts
y_pred = best_model.predict(X_eval)  # predicted class indices (0/1/2)

n_samples = X_eval.shape[0]
n_features = X_eval.shape[1]
shap_pred = np.zeros((n_samples, n_features), dtype=float)

for c in range(len(class_names)):
    mask = (y_pred == c)
    if not np.any(mask):
        continue
    shap_c = shap_array_for_class(sv, c)  # (n_samples, n_features)
    # same indexing because sv was computed on X_eval in the same order
    shap_pred[mask] = shap_c[mask]

print("shap_pred shape:", shap_pred.shape)  # (n_samples, n_features)

# 3) PCA: high-dimensional SHAP vectors -> 2D
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(shap_pred)  # (n_samples, 2)

# 4) Plot, colored by TRUE label (NeuronType)
colors = ["tab:red", "tab:blue", "tab:green"][:len(class_names)]

plt.figure(figsize=(6, 5))
for k, cname in enumerate(class_names):
    mask = (y_true == k)
    plt.scatter(
        Z[mask, 0],
        Z[mask, 1],
        s=35,
        alpha=0.8,
        c=colors[k],
        label=cname
    )

plt.xlabel("PC1 (SHAP space)")
plt.ylabel("PC2 (SHAP space)")
plt.title(f"{best_name}: PCA on SHAP values (predicted class)")
plt.legend()
plt.tight_layout()
plt.show()


## What its doing above
1. What SHAP is doing in your notebook

You have a multiclass tree model (XGBoost). For each sample 𝑥x and class 𝑘 k:
- The model produces a raw output 𝑓𝑘(𝑥) (for XGBoost this is usually log-odds / logit for class k).

